In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.table("databricks_aws_lakehouse.bronze.bz_superstore")

In [0]:
df.display()

In [0]:
df = df.drop("_corrupted_data")

df.limit(5).display()

In [0]:
df_silver = df.withColumn("sales",col("sales").cast("double"))\
                .withColumn("qualtity",col("qualtity").cast("double"))\
                .withColumn("discount",col("discount").cast("double"))\
                .withColumn("profit",col("profit").cast("double"))
df_silver.display()

In [0]:
df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
df_silver.filter(col('sales').isNull()).display()

**ROW LEVEL idempotency**

In [0]:
if not spark.catalog.tableExists("databricks_aws_lakehouse.silver.silver_enr"):
    df_silver.dropDuplicates()\
        .write.mode("overwrite")\
        .saveAsTable("databricks_aws_lakehouse.silver.silver_enr")

In [0]:
%sql
select * from databricks_aws_lakehouse.silver.silver_enr limit 5